In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#%matplotlib inline

#import seaborn as sns


import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
# from celloracle import motif_analysis as ma
# import celloracle as co
import scanpy as sc


In [ ]:
if os.path.exists('/data2st2/junyi/output/stg1028/CUMS_4VN/CUMS_4VN_bulk.h5ad'):
    adata_sc = sc.read_h5ad('/data2st2/junyi/output/stg1028/CUMS_4VN/CUMS_4VN.h5ad')
else:
    adata_sc = sc.read_h5ad('/data2st2/junyi/output/stg1028/CUMS_4VN/CUMS_4VN.h5ad')
    adata_sc_M = adata_sc[adata_sc.obs['sex'] == 'M']
    adata_3REGION = adata_sc_M[adata_sc_M.obs['region'].isin(['PFC','HPF','AMY'])]
    adata_3REGION.obs['condition'] = adata_3REGION.obs['status'].map({'CON':'MW','SUS':'MC'})
    adata_3REGION.obs['region_celltypeL2_condition'] = adata_3REGION.obs['region'].astype(str) + "_" + adata_3REGION.obs['celltype.L2'].astype(str) + '_' + adata_3REGION.obs['condition'].astype(str)
    adata_3REGION.obs['region_celltypeL2_condition']=adata_3REGION.obs['region_celltypeL2_condition'].replace('/','-',regex=True)
    adata_3REGION.obs['region_celltypeL2_condition']=adata_3REGION.obs['region_celltypeL2_condition'].replace(' ','_',regex=True)
    adata_3REGION.obs['region_celltypeL2_condition']=adata_3REGION.obs['region_celltypeL2_condition'].str.replace('Cajal?Retzius','Cajal-Retzius')
    adata_3REGION_bulk = sc.get.aggregate(adata_3REGION, by='region_celltypeL2_condition', layer='counts',func='sum')
    adata_3REGION_bulk.X=adata_3REGION_bulk.layers['sum'].copy()
    adata_3REGION_bulk['region_celltypeL2_condition'].replace('HIP','HPF',regex=True)
    adata_3REGION_bulk.obs['region_celltypeL2_condition'] = adata_3REGION_bulk.obs['region_celltypeL2_condition'].str.replace('?', '-', regex=False)
    adata_3REGION_bulk.obs['region'] = adata_3REGION_bulk.obs['region_celltypeL2_condition'].apply(lambda x: x.split('_')[0])
    adata_3REGION_bulk.obs['celltype.L2'] = adata_3REGION_bulk.obs['region_celltypeL2_condition'].apply(lambda x: '_'.join(x.split('_')[1:-1]))
    adata_3REGION_bulk.obs['condition'] = adata_3REGION_bulk.obs['region_celltypeL2_condition'].apply(lambda x: x.split('_')[-1])
    adata_3REGION_bulk.write_h5ad('/data2st2/junyi/output/stg1028/CUMS_4VN/CUMS_4VN_bulk.h5ad')

In [ ]:
if os.path.exists('/data2st1/junyi/output/atac1112/3REGIONS_peak_l2_aggregated.h5ad'):
    adata_3REGION_atac = sc.read_h5ad('/data2st1/junyi/output/atac1112/3REGIONS_peak_l2_aggregated.h5ad')
else:
    adata_atac = sc.read_h5ad('/data2st1/junyi/output/atac1112/3REGIONS_peak.h5ads')
    adata_atac.obs['region_celltypeL2_condition'] = adata_atac.obs['Region'].astype(str) + "_" + adata_atac.obs['celltype.L2.refined'].astype(str) + '_' + adata_atac.obs['expriment'].astype(str)
    adata_atac
    adata_atac.obs['celltype.L2'] = adata_atac.obs['region_celltypeL2_condition'].apply(lambda x: '_'.join(x.split('_')[1:-1]))
    adata_atac.obs['condition'] = adata_atac.obs['region_celltypeL2_condition'].apply(lambda x: x.split('_')[-1])
    adata_atac.obs['region_celltypeL2_condition']=adata_atac.obs['region_celltypeL2_condition'].replace('/','-',regex=True)
    adata_atac.obs['region_celltypeL2_condition']=adata_atac.obs['region_celltypeL2_condition'].replace(' ','_',regex=True)
    adata_3REGION_atac = sc.get.aggregate(adata_atac, by='region_celltypeL2_condition', layer='count',func='sum')
    adata_3REGION_atac.obs['region_celltypeL2_condition'] = adata_3REGION_atac.obs['region_celltypeL2_condition'].str.replace('HIP','HPF')
    adata_3REGION_bulk.obs
    set(adata_3REGION_atac.obs['region_celltypeL2_condition']).difference(set(adata_3REGION_bulk.obs['region_celltypeL2_condition']))
    # adata_3REGION_bulk中region_celltypeL2_condition把未知符号改成"-"

    set(adata_3REGION_bulk.obs['region_celltypeL2_condition']).difference(set(adata_3REGION_atac.obs['region_celltypeL2_condition']))
    adata_3REGION_atac.write_h5ad('/data2st1/junyi/output/atac1112/3REGIONS_peak_l2_aggregated.h5ad')

In [147]:
adata_3REGION_atac.var

""
chr1:3003518-3004019
chr1:3007481-3007982
chr1:3012480-3012981
chr1:3013424-3013925
chr1:3014717-3015218
...
chrY:90812660-90813161
chrY:90811433-90811934
chrY:90813596-90814097
chrY:90825201-90825702


In [150]:
region_parts = adata_3REGION_atac.var_names.to_series().astype(str).str.extract(r'^(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)$')
adata_3REGION_atac.var['chr'] = region_parts['chr'].values
adata_3REGION_atac.var['start'] = pd.to_numeric(region_parts['start'], errors='coerce')
adata_3REGION_atac.var['end'] = pd.to_numeric(region_parts['end'], errors='coerce')

In [146]:
annotations = [
    "/data2st1/junyi/generegion_vM23/promoter2k.bed",
]

In [158]:
df_peaks = adata_3REGION_atac.var

In [154]:
df_promoters = pd.read_csv(annotations[0], sep='\t', header=None, names=['chr', 'start', 'end','score','strand','gene_name', 'gene_id','cCRE'])

In [155]:
max_dist = 500

,chr,start,end,score,strand,gene_name,gene_id,cCRE
0,chr1,3671348,3673348,.,-,Xkr4,ENSMUSG00000051951.5,promoter
1,chr1,4352825,4354825,.,-,Rp1,ENSMUSG00000025900.12,promoter
2,chr1,4409187,4411187,.,-,Rp1,ENSMUSG00000025900.12,promoter
3,chr1,4409187,4411187,.,-,Rp1,ENSMUSG00000025900.12,promoter
4,chr1,4492591,4494591,.,-,Sox17,ENSMUSG00000025902.13,promoter
...,...,...,...,...,...,...,...,...
59906,chrY,89052243,89054243,.,+,Gm21294,ENSMUSG00000102045.1,promoter
59907,chrY,89744111,89746111,.,-,Gm21996,ENSMUSG00000100608.1,promoter
59908,chrY,90400678,90402678,.,+,Gm20837,ENSMUSG00000096178.7,promoter
59909,chrY,90400687,90402687,.,+,Gm20837,ENSMUSG00000096178.7,promoter


In [161]:
df_deg = pd.read_csv("/data2st1/junyi/output/atac1112/dar/celltype.L2/mast_ngsa_noco_degs_fdr_log2fc0_filtered.csv")
df_deg['Region Subclass'] = df_deg['Region subclass'].str.replace(" ",'_')
df_deg['Region Subclass'] = df_deg['Region Subclass'].str.replace("/",'-')
df_deg = df_deg[df_deg.Sex=='M']
df_deg = df_deg[df_deg.Region.isin(['HPF','PFC','AMY'])]
df_promoters_deg = df_promoters[df_promoters['gene_name'].isin(df_deg.Gene.unique())].copy()
df_promoters_deg = df_promoters_deg.drop_duplicates()
df_promoters_deg

,chr,start,end,peak_id
0,chr1,3003518,3004019,chr1:3003518-3004019
1,chr1,3007481,3007982,chr1:3007481-3007982
2,chr1,3012480,3012981,chr1:3012480-3012981
3,chr1,3013424,3013925,chr1:3013424-3013925
4,chr1,3014717,3015218,chr1:3014717-3015218
...,...,...,...,...
1660761,chrY,90812660,90813161,chrY:90812660-90813161
1660762,chrY,90811433,90811934,chrY:90811433-90811934
1660763,chrY,90813596,90814097,chrY:90813596-90814097
1660764,chrY,90825201,90825702,chrY:90825201-90825702


In [197]:
df_deg

,Gene,pval,log2FC,ci.hi,ci.lo,FDR,Bonferroni,Subclass,Region,Sex,Method,Ensemble,Gene_name,Direction,Neurotransmitter,Region subclass,Region Subclass
175654,1810062O18Rik,5.808074e-03,0.070354,0.120168,0.020540,0.040524,1.000000,Astrocyte-2,PFC,M,MAST,ENSMUSG00000084925,"""1810062O18Rik""",Up,NN,PFC Astrocyte-2,PFC_Astrocyte-2
175655,2610035D17Rik,7.490775e-04,0.088500,0.139018,0.037981,0.009548,1.000000,Astrocyte-2,PFC,M,MAST,ENSMUSG00000087259,"""2610035D17Rik""",Up,NN,PFC Astrocyte-2,PFC_Astrocyte-2
175656,4732471J01Rik,8.759375e-04,0.053831,0.097587,0.010075,0.010621,1.000000,Astrocyte-2,PFC,M,MAST,ENSMUSG00000053714,"""4732471J01Rik""",Up,NN,PFC Astrocyte-2,PFC_Astrocyte-2
175657,4833420G17Rik,4.983856e-03,0.040075,0.087102,-0.006953,0.036435,1.000000,Astrocyte-2,PFC,M,MAST,ENSMUSG00000062822,"""4833420G17Rik""",Up,NN,PFC Astrocyte-2,PFC_Astrocyte-2
175658,4930402H24Rik,7.804847e-04,0.105143,0.164551,0.045736,0.009823,1.000000,Astrocyte-2,PFC,M,MAST,ENSMUSG00000027309,"""4930402H24Rik""",Up,NN,PFC Astrocyte-2,PFC_Astrocyte-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355341,Zswim6,4.582351e-03,-0.098474,-0.027546,-0.169402,0.039133,1.000000,OPC,AMY,M,MAST,ENSMUSG00000032846,"""Zswim6""",Down,NN,AMY OPC,AMY_OPC
355342,Apoe,4.143067e-07,0.876975,1.205846,0.548104,0.000844,0.001689,VLMC,AMY,M,MAST,ENSMUSG00000002985,"""Apoe""",Up,NN,AMY VLMC,AMY_VLMC
355343,Meg3,3.685718e-07,0.896927,1.220253,0.573601,0.000844,0.001502,VLMC,AMY,M,MAST,ENSMUSG00000021268,"""Meg3""",Up,NN,AMY VLMC,AMY_VLMC
355344,Slc6a13,3.642790e-06,0.890118,1.255269,0.524966,0.004949,0.014848,VLMC,AMY,M,MAST,ENSMUSG00000030108,"""Slc6a13""",Up,NN,AMY VLMC,AMY_VLMC


In [174]:
import subprocess
import tempfile
from pathlib import Path

peak_table = adata_3REGION_atac.var.copy()
peak_table['peak_id'] = peak_table.index.astype(str)
peak_table = peak_table.reset_index(drop=True)
peak_table['start'] = pd.to_numeric(peak_table['start'], errors='coerce')
peak_table['end'] = pd.to_numeric(peak_table['end'], errors='coerce')
peak_table = peak_table.dropna(subset=['chr', 'start', 'end']).copy()
peak_table['start'] = peak_table['start'].astype(int)
peak_table['end'] = peak_table['end'].astype(int)

promoter_table = df_promoters_deg.copy()
promoter_table['promoter_id'] = (
    promoter_table['chr'].astype(str) + ':' +
    promoter_table['start'].astype(str) + '-' +
    promoter_table['end'].astype(str)
    )
promoter_table['start'] = pd.to_numeric(promoter_table['start'], errors='coerce')
promoter_table['end'] = pd.to_numeric(promoter_table['end'], errors='coerce')
promoter_table = promoter_table.dropna(subset=['chr', 'start', 'end']).copy()
promoter_table['start'] = promoter_table['start'].astype(int)
promoter_table['end'] = promoter_table['end'].astype(int)

promoter_bed = promoter_table[['chr', 'start', 'end', 'promoter_id', 'gene_name', 'gene_id', 'cCRE']].sort_values(['chr', 'start', 'end'])
peak_bed = peak_table[['chr', 'start', 'end', 'peak_id']].sort_values(['chr', 'start', 'end'])


In [228]:
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    promoter_bed_path = tmpdir / 'promoters.bed'
    peak_bed_path = tmpdir / 'peaks.bed'
    output_path = tmpdir / 'promoter_peak_500kb.bed'

    promoter_bed.to_csv(promoter_bed_path, sep='\t', header=False, index=False)
    peak_bed.to_csv(peak_bed_path, sep='\t', header=False, index=False)

    cmd = [
        'bedtools', 'window',
        '-a', str(promoter_bed_path),
        '-b', str(peak_bed_path),
        '-w', '500000'
    ]

    with open(output_path, 'w') as outfile:
        subprocess.run(cmd, check=True, stdout=outfile)

    df_promoter_peak_500kb = pd.read_csv(
        output_path,
        sep='\t',
        header=None,
        names=[
            'chrom', 'promoter_start', 'promoter_end', 'promoter_id', 'gene_name', 'gene_id', 'cCRE',
            'peak_chrom', 'peak_start', 'peak_end', 'peak_id'
        ]
    )

df_promoter_peak_500kb['distance'] = np.where(
    df_promoter_peak_500kb['peak_end'] < df_promoter_peak_500kb['promoter_start'],
    df_promoter_peak_500kb['promoter_start'] - df_promoter_peak_500kb['peak_end'],
    np.where(
        df_promoter_peak_500kb['peak_start'] > df_promoter_peak_500kb['promoter_end'],
        df_promoter_peak_500kb['peak_start'] - df_promoter_peak_500kb['promoter_end'],
        0
    )
)

df_promoter_peak_500kb = df_promoter_peak_500kb[[
    'promoter_id', 'gene_name', 'gene_id', 'cCRE',
    'chrom', 'promoter_start', 'promoter_end',
    'peak_id', 'peak_chrom', 'peak_start', 'peak_end', 'distance'
]]

df_promoter_peak_500kb

,promoter_id,gene_name,gene_id,cCRE,chrom,promoter_start,promoter_end,peak_id,peak_chrom,peak_start,peak_end,distance
0,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3171467-3171968,chr1,3171467,3171968,499380
1,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3174053-3174554,chr1,3174053,3174554,496794
2,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3174697-3175198,chr1,3174697,3175198,496150
3,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3177372-3177873,chr1,3177372,3177873,493475
4,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3181089-3181590,chr1,3181089,3181590,489758
...,...,...,...,...,...,...,...,...,...,...,...,...
9640198,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169639840-169640341,chrX,169639840,169640341,337134
9640199,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169787102-169787603,chrX,169787102,169787603,189872
9640200,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169934728-169935229,chrX,169934728,169935229,42246
9640201,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169950963-169951464,chrX,169950963,169951464,26011


In [178]:
df_promoter_peak_500kb.to_csv("/data2st1/junyi/output/atac1112/3REGIONS_peak_l2_promoter_peak_500kb.csv", index=False)

In [181]:
sc.pp.normalize_total(adata_3REGION_bulk, target_sum=1e6)

AxisError: axis 1 is out of bounds for array of dimension 0

In [183]:
adata_3REGION_bulk.X.max()

337172.4943995325

In [190]:
adata_3REGION_bulk.X

array([[7.11807715, 5.25800961, 3.95225592, ..., 3.7312332 , 0.98352228,
        5.82881677],
       [7.19702418, 5.23690737, 4.43945408, ..., 3.78113671, 0.78929655,
        5.82517087],
       [7.56839773, 5.55943757, 4.64514178, ..., 3.67802292, 0.6971239 ,
        5.79270975],
       ...,
       [4.59662652, 0.        , 1.96491828, ..., 4.53276016, 0.        ,
        5.22051736],
       [4.74048104, 2.93027052, 1.51436639, ..., 4.41365136, 0.        ,
        4.82934426],
       [0.        , 0.        , 0.        , ..., 6.0944272 , 0.        ,
        0.        ]])

In [225]:
adata_3REGION_atac.X

AnnData object with n_obs × n_vars = 223 × 1660766
    obs: 'region_celltypeL2_condition'
    var: 'chr', 'start', 'end'
    uns: 'log1p'
    layers: 'sum'

In [186]:
adata_3REGION_bulk.layers['sum'].max()

8438345.0

In [182]:
adata_3REGION_atac.X = adata_3REGION_atac.layers['sum'].copy()
sc.pp.normalize_total(adata_3REGION_atac, target_sum=1e6)

In [187]:
sc.pp.log1p(adata_3REGION_bulk)
sc.pp.log1p(adata_3REGION_atac)

In [226]:
adata_3REGION_bulk.write_h5ad('/data2st2/junyi/output/stg1028/CUMS_4VN/CUMS_4VN_bulk.h5ad')
adata_3REGION_atac.write_h5ad('/data2st1/junyi/output/atac1112/3REGIONS_peak_l2_aggregated.h5ad')

In [193]:
# subset overlapping obs for adata_3REGION_bulk and adata_3REGION_atac
common_obs = set(adata_3REGION_bulk.obs_names).intersection(set(adata_3REGION_atac.obs_names))
adata_3REGION_bulk_sub = adata_3REGION_bulk[adata_3REGION_bulk.obs_names.isin(common_obs)].copy()
adata_3REGION_atac_sub = adata_3REGION_atac[adata_3REGION_atac.obs_names.isin(common_obs)].copy()

In [ ]:
adata_3REGION_bulk_sub_deg = adata_3REGION_bulk_sub[:,adata_3REGION_bulk_sub.var.index.isin(df_deg['Gene'].unique())]

In [203]:
adata_3REGION_bulk_sub_deg

View of AnnData object with n_obs × n_vars = 147 × 9424
    obs: 'region_celltypeL2_condition', 'region', 'celltype.L2', 'condition'
    var: 'gene_ids'
    uns: 'log1p'
    layers: 'sum'

In [196]:
adata_3REGION_atac_sub

AnnData object with n_obs × n_vars = 147 × 1660766
    obs: 'region_celltypeL2_condition'
    var: 'chr', 'start', 'end'
    uns: 'log1p'
    layers: 'sum'

In [192]:
df_promoter_peak_500kb

,promoter_id,gene_name,gene_id,cCRE,chrom,promoter_start,promoter_end,peak_id,peak_chrom,peak_start,peak_end,distance
0,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3171467-3171968,chr1,3171467,3171968,499380
1,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3174053-3174554,chr1,3174053,3174554,496794
2,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3174697-3175198,chr1,3174697,3175198,496150
3,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3177372-3177873,chr1,3177372,3177873,493475
4,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3181089-3181590,chr1,3181089,3181590,489758
...,...,...,...,...,...,...,...,...,...,...,...,...
9640198,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169639840-169640341,chrX,169639840,169640341,337134
9640199,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169787102-169787603,chrX,169787102,169787603,189872
9640200,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169934728-169935229,chrX,169934728,169935229,42246
9640201,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169950963-169951464,chrX,169950963,169951464,26011


In [216]:
df_promoter_peak_500kbselected =    df_promoter_peak_500kb[df_promoter_peak_500kb.distance>0]

(9612666, 12)

In [251]:
df_promoter_peak_500kbselected

,promoter_id,gene_name,gene_id,cCRE,chrom,promoter_start,promoter_end,peak_id,peak_chrom,peak_start,peak_end,distance
0,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3171467-3171968,chr1,3171467,3171968,499380
1,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3174053-3174554,chr1,3174053,3174554,496794
2,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3174697-3175198,chr1,3174697,3175198,496150
3,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3177372-3177873,chr1,3177372,3177873,493475
4,chr1:3671348-3673348,Xkr4,ENSMUSG00000051951.5,promoter,chr1,3671348,3673348,chr1:3181089-3181590,chr1,3181089,3181590,489758
...,...,...,...,...,...,...,...,...,...,...,...,...
9640198,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169639840-169640341,chrX,169639840,169640341,337134
9640199,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169787102-169787603,chrX,169787102,169787603,189872
9640200,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169934728-169935229,chrX,169934728,169935229,42246
9640201,chrX:169977475-169979475,Mid1,ENSMUSG00000035299.16,promoter,chrX,169977475,169979475,chrX:169950963-169951464,chrX,169950963,169951464,26011


In [217]:
from scipy.stats import pearsonr
from scipy import sparse
from statsmodels.stats.multitest import multipletests

common_obs_sorted = sorted(set(adata_3REGION_bulk.obs_names).intersection(set(adata_3REGION_atac.obs_names)))
rna_corr = adata_3REGION_bulk[common_obs_sorted, :].copy()
atac_corr = adata_3REGION_atac[common_obs_sorted, :].copy()

candidate_pairs = df_promoter_peak_500kbselected[['peak_id', 'gene_name']].drop_duplicates().copy()
candidate_pairs = candidate_pairs[
    candidate_pairs['peak_id'].isin(atac_corr.var_names) &
    candidate_pairs['gene_name'].isin(rna_corr.var_names)
].copy()

def _to_dense_vector(x):
    if sparse.issparse(x):
        return np.asarray(x.toarray()).ravel()
    return np.asarray(x).ravel()

peak_index = pd.Index(atac_corr.var_names)
gene_index = pd.Index(rna_corr.var_names)

In [218]:
candidate_pairs

,peak_id,gene_name
0,chr1:3171467-3171968,Xkr4
1,chr1:3174053-3174554,Xkr4
2,chr1:3174697-3175198,Xkr4
3,chr1:3177372-3177873,Xkr4
4,chr1:3181089-3181590,Xkr4
...,...,...
9639774,chrX:169934728-169935229,Mid1
9639775,chrX:169950963-169951464,Mid1
9639776,chrX:169983775-169984276,Mid1
9639880,chrX:169798871-169799372,Mid1


In [219]:
from concurrent.futures import ThreadPoolExecutor

peak_names = candidate_pairs['peak_id'].unique()
gene_names = candidate_pairs['gene_name'].unique()

peak_matrix = atac_corr[:, peak_names].X
gene_matrix = rna_corr[:, gene_names].X

if sparse.issparse(peak_matrix):
    peak_matrix = peak_matrix.toarray()
else:
    peak_matrix = np.asarray(peak_matrix)

if sparse.issparse(gene_matrix):
    gene_matrix = gene_matrix.toarray()
else:
    gene_matrix = np.asarray(gene_matrix)

peak_value_map = {peak: peak_matrix[:, idx] for idx, peak in enumerate(peak_names)}
gene_value_map = {gene: gene_matrix[:, idx] for idx, gene in enumerate(gene_names)}

def process_candidate_chunk(chunk_df):
    chunk_results = []
    for row in chunk_df.itertuples(index=False):
        peak_values = np.asarray(peak_value_map[row.peak_id]).ravel()
        gene_values = np.asarray(gene_value_map[row.gene_name]).ravel()

        valid_mask = np.isfinite(peak_values) & np.isfinite(gene_values)
        peak_values = peak_values[valid_mask]
        gene_values = gene_values[valid_mask]

        if len(peak_values) < 3:
            continue
        if np.all(peak_values == peak_values[0]) or np.all(gene_values == gene_values[0]):
            continue

        corr, pval = pearsonr(peak_values, gene_values)
        chunk_results.append({
            'peak_id': row.peak_id,
            'gene_name': row.gene_name,
            'correlation': corr,
            'pval': pval,
            'n_samples': len(peak_values)
        })
    return chunk_results



,peak_id,gene_name,correlation,pval,n_samples,FDR
0,chr1:6486933-6487434,St18,0.813606,5.745271e-36,147,5.745271e-31
1,chr1:55225272-55225773,Rftn2,0.781948,1.448553e-31,147,7.242767e-27
2,chr1:9299540-9300041,Sntg1,0.758464,9.479920e-29,147,3.159973e-24
3,chr1:24587192-24587693,Col19a1,0.755510,2.033642e-28,147,5.084105e-24
4,chr1:6594381-6594882,St18,0.747499,1.527030e-27,147,3.054060e-23
...,...,...,...,...,...,...
23738,chr1:59414860-59415361,Nop58,0.207071,1.185265e-02,147,4.992853e-02
23739,chr1:56712264-56712765,Satb2,0.207070,1.185303e-02,147,4.992853e-02
23740,chr1:11226960-11227461,Prex2,0.207065,1.185498e-02,147,4.993462e-02
23741,chr1:31840640-31841141,Khdrbs2,0.207047,1.186261e-02,147,4.996465e-02


In [220]:
candidate_chunks = [chunk.copy() for chunk in np.array_split(candidate_pairs, 40) if len(chunk) > 0]

corr_results = []
with ThreadPoolExecutor(max_workers=min(20, len(candidate_chunks))) as executor:
    for chunk_result in executor.map(process_candidate_chunk, candidate_chunks):
        corr_results.extend(chunk_result)

df_peak_gene_corr = pd.DataFrame(corr_results)
if not df_peak_gene_corr.empty:
    df_peak_gene_corr['FDR'] = multipletests(df_peak_gene_corr['pval'], method='fdr_bh')[1]
else:
    df_peak_gene_corr['FDR'] = pd.Series(dtype=float)



,peak_id,gene_name,correlation,pval,n_samples,FDR
0,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54
1,chr15:79164315-79164816,Sox10,0.901283,1.545839e-54,147,5.240049e-48
2,chr8:111375106-111375607,Fa2h,0.893897,2.197761e-52,147,4.966613e-46
3,chr3:125898900-125899401,Ugt8a,0.880538,7.237274e-49,147,1.226637e-42
4,chr12:80112823-80113324,Zfp36l1,0.874969,1.597721e-47,147,2.166367e-41
...,...,...,...,...,...,...
1587341,chr16:76592070-76592571,Usp25,-0.207422,1.170661e-02,147,4.999906e-02
1587342,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02
1587343,chr16:11001653-11002154,Rsl1d1,-0.207422,1.170677e-02,147,4.999964e-02
1587344,chr1:135342876-135343377,Shisa4,0.207422,1.170679e-02,147,4.999969e-02


In [269]:
df_peak_gene_corr_sig = df_peak_gene_corr[
    df_peak_gene_corr['FDR'] < 0.05
].sort_values(['FDR', 'pval', 'correlation'], ascending=[True, True, False]).reset_index(drop=True)

df_peak_gene_corr_sig

,peak_id,gene_name,correlation,pval,n_samples,FDR
0,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54
1,chr15:79164315-79164816,Sox10,0.901283,1.545839e-54,147,5.240049e-48
2,chr8:111375106-111375607,Fa2h,0.893897,2.197761e-52,147,4.966613e-46
3,chr3:125898900-125899401,Ugt8a,0.880538,7.237274e-49,147,1.226637e-42
4,chr12:80112823-80113324,Zfp36l1,0.874969,1.597721e-47,147,2.166367e-41
...,...,...,...,...,...,...
1587341,chr16:76592070-76592571,Usp25,-0.207422,1.170661e-02,147,4.999906e-02
1587342,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02
1587343,chr16:11001653-11002154,Rsl1d1,-0.207422,1.170677e-02,147,4.999964e-02
1587344,chr1:135342876-135343377,Shisa4,0.207422,1.170679e-02,147,4.999969e-02


In [223]:
df_peak_gene_corr_sig.to_csv("/data2st1/junyi/output/atac1112/cCRE/3REGIONS_DEG_correlation_sig.csv", index=False)

In [224]:
df_peak_gene_corr.to_csv("/data2st1/junyi/output/atac1112/cCRE/3REGIONS_DEG_correlation.csv", index=False)

In [270]:
df_meta=df_promoter_peak_500kbselected[['promoter_id','peak_id','gene_name','distance']]

In [271]:
df_meta

,promoter_id,peak_id,gene_name,distance
0,chr1:3671348-3673348,chr1:3171467-3171968,Xkr4,499380
1,chr1:3671348-3673348,chr1:3174053-3174554,Xkr4,496794
2,chr1:3671348-3673348,chr1:3174697-3175198,Xkr4,496150
3,chr1:3671348-3673348,chr1:3177372-3177873,Xkr4,493475
4,chr1:3671348-3673348,chr1:3181089-3181590,Xkr4,489758
...,...,...,...,...
9640198,chrX:169977475-169979475,chrX:169639840-169640341,Mid1,337134
9640199,chrX:169977475-169979475,chrX:169787102-169787603,Mid1,189872
9640200,chrX:169977475-169979475,chrX:169934728-169935229,Mid1,42246
9640201,chrX:169977475-169979475,chrX:169950963-169951464,Mid1,26011


In [274]:
df_peak_gene_corr_sig

,peak_id,gene_name,correlation,pval,n_samples,FDR
0,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54
1,chr15:79164315-79164816,Sox10,0.901283,1.545839e-54,147,5.240049e-48
2,chr8:111375106-111375607,Fa2h,0.893897,2.197761e-52,147,4.966613e-46
3,chr3:125898900-125899401,Ugt8a,0.880538,7.237274e-49,147,1.226637e-42
4,chr12:80112823-80113324,Zfp36l1,0.874969,1.597721e-47,147,2.166367e-41
...,...,...,...,...,...,...
1587341,chr16:76592070-76592571,Usp25,-0.207422,1.170661e-02,147,4.999906e-02
1587342,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02
1587343,chr16:11001653-11002154,Rsl1d1,-0.207422,1.170677e-02,147,4.999964e-02
1587344,chr1:135342876-135343377,Shisa4,0.207422,1.170679e-02,147,4.999969e-02


In [273]:
pd.merge(df_peak_gene_corr_sig, df_meta, on=['peak_id', 'gene_name'], how='inner')

,peak_id,gene_name,correlation,pval,n_samples,FDR,promoter_id,distance
0,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54,chr15:79159348-79161348,51426
1,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54,chr15:79163716-79165716,47058
2,chr15:79164315-79164816,Sox10,0.901283,1.545839e-54,147,5.240049e-48,chr15:79159348-79161348,2967
3,chr8:111375106-111375607,Fa2h,0.893897,2.197761e-52,147,4.966613e-46,chr8:111393752-111395752,18145
4,chr3:125898900-125899401,Ugt8a,0.880538,7.237274e-49,147,1.226637e-42,chr3:125915459-125917459,16058
...,...,...,...,...,...,...,...,...
2319517,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02,chr19:38124629-38126629,74483
2319518,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02,chr19:38125068-38127068,74922
2319519,chr16:11001653-11002154,Rsl1d1,-0.207422,1.170677e-02,147,4.999964e-02,chr16:11203255-11205255,201101
2319520,chr1:135342876-135343377,Shisa4,0.207422,1.170679e-02,147,4.999969e-02,chr1:135374961-135376961,31584


In [266]:
df_peak_gene_corr_sig

,peak_id,gene_name,correlation,pval,n_samples,FDR,promoter_id,distance
0,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54,chr15:79159348-79161348,51426
1,chr15:79212774-79213275,Sox10,0.920316,5.623160e-61,147,3.812251e-54,chr15:79163716-79165716,47058
2,chr15:79164315-79164816,Sox10,0.901283,1.545839e-54,147,5.240049e-48,chr15:79159348-79161348,2967
3,chr8:111375106-111375607,Fa2h,0.893897,2.197761e-52,147,4.966613e-46,chr8:111393752-111395752,18145
4,chr3:125898900-125899401,Ugt8a,0.880538,7.237274e-49,147,1.226637e-42,chr3:125915459-125917459,16058
...,...,...,...,...,...,...,...,...
2319517,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02,chr19:38124629-38126629,74483
2319518,chr19:38049645-38050146,Rbp4,0.207422,1.170667e-02,147,4.999925e-02,chr19:38125068-38127068,74922
2319519,chr16:11001653-11002154,Rsl1d1,-0.207422,1.170677e-02,147,4.999964e-02,chr16:11203255-11205255,201101
2319520,chr1:135342876-135343377,Shisa4,0.207422,1.170679e-02,147,4.999969e-02,chr1:135374961-135376961,31584


In [316]:
df_dar = pd.read_csv("/data2st1/junyi/output/atac1112/dar/celltype.L2/MAST_dar_annotated.csv")

In [275]:
df_peak_gene_corr_sig.rename(columns={'peak_id':'names','gene_name':'gene_coaccess'}, inplace=True)

In [276]:
df_peak_gene_select = df_peak_gene_corr_sig[['names','gene_coaccess','correlation']].copy()

In [277]:
df_dar.primary_region.value_counts()

promoter      191885
intron         79114
downstream     33601
distal         26796
exon           20952
UTR            20287
genebody        5353
Name: primary_region, dtype: int64

In [ ]:
df_dar_np = df_dar[~df_dar.primary_region.isin(['dstal', 'downstream','intron'])]

In [278]:
df_dar_merge = df_dar.merge(df_peak_gene_select, left_on='names', right_on='names', how='left')

In [280]:
df_full = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE.peak_gene_corr.sig.csv')

In [282]:
df_full.rename(columns={'peak_id':'names','gene_name':'gene_coaccess'}, inplace=True)

In [285]:
df_peak_gene_select = df_full[['names','gene_coaccess','correlation']].copy()

In [318]:
df_dar_merge = df_dar.merge(df_peak_gene_select, left_on='names', right_on='names', how='inner')

In [319]:
df_dar_merge['gene_back']= df_dar_merge['gene_name']

In [320]:
df_dar_merge['gene_name'] = df_dar_merge['gene_coaccess']

In [321]:
df_dar_merge

,names,Pr(>Chisq),coef,ci.hi,ci.lo,padjust,celltype.L2,region,ctname,fdr,...,Ensemble,Gene,log2FC,Direction,Neurotransmitter,Gene_name,FDR,gene_coaccess,correlation,gene_back
0,chr10:100456562-100457063,4.696654e-03,0.002657,0.014730,-0.009417,1.000000e+00,HPF_CA3_Glut,HPF,HPF_CA3_Glut,8.837957e-03,...,ENSMUSG00000036676.14,chr10:100456562-100457063,0.002657,Up,Glut,chr10:100456562-100457063,8.837957e-03,Kitl,0.266765,Tmtc3
1,chr10:100456562-100457063,4.696654e-03,0.002657,0.014730,-0.009417,1.000000e+00,HPF_CA3_Glut,HPF,HPF_CA3_Glut,8.837957e-03,...,ENSMUSG00000036676.14,chr10:100456562-100457063,0.002657,Up,Glut,chr10:100456562-100457063,8.837957e-03,4930430F08Rik,0.221632,Tmtc3
2,chr10:100487209-100487710,4.226688e-12,0.080716,0.103431,0.058001,1.371518e-07,HPF_CA3_Glut,HPF,HPF_CA3_Glut,1.878792e-09,...,ENSMUSG00000036676.14,chr10:100487209-100487710,0.080716,Up,Glut,chr10:100487209-100487710,1.878792e-09,Cep290,0.225227,Tmtc3
3,chr10:100487209-100487710,4.226688e-12,0.080716,0.103431,0.058001,1.371518e-07,HPF_CA3_Glut,HPF,HPF_CA3_Glut,1.878792e-09,...,ENSMUSG00000036676.14,chr10:100487209-100487710,0.080716,Up,Glut,chr10:100487209-100487710,1.878792e-09,Cep290,0.225227,Tmtc3
4,chr10:100487209-100487710,6.084578e-06,0.034846,0.050596,0.019097,5.822942e-02,AMY_Meis1_Abi3bp_Glut,AMY,AMY_Meis1_Abi3bp_Glut,1.621989e-04,...,ENSMUSG00000036676.14,chr10:100487209-100487710,0.034846,Up,Glut,chr10:100487209-100487710,1.621989e-04,Cep290,0.225227,Tmtc3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
879233,chrX:73824735-73825236,4.155088e-03,0.013819,0.025836,0.001803,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,1.786323e-02,...,ENSMUSG00000002006.12,chrX:73824735-73825236,0.013819,Up,Glut,chrX:73824735-73825236,1.786323e-02,Gdi1,0.273216,Pdzd4
879234,chrX:73824735-73825236,4.155088e-03,0.013819,0.025836,0.001803,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,1.786323e-02,...,ENSMUSG00000002006.12,chrX:73824735-73825236,0.013819,Up,Glut,chrX:73824735-73825236,1.786323e-02,Idh3g,0.262363,Pdzd4
879235,chrX:73824735-73825236,4.155088e-03,0.013819,0.025836,0.001803,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,1.786323e-02,...,ENSMUSG00000002006.12,chrX:73824735-73825236,0.013819,Up,Glut,chrX:73824735-73825236,1.786323e-02,Pnck,0.243705,Pdzd4
879236,chrX:99505795-99506296,6.136253e-03,0.013835,0.026267,0.001404,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,2.356687e-02,...,ENSMUSG00000034403.16,chrX:99505795-99506296,0.013835,Up,Glut,chrX:99505795-99506296,2.356687e-02,Stard8,-0.471481,Pja1


In [314]:
df_dar_merge.dropna(subset=['coef'], inplace=True)

In [304]:
df_dar_merge['regulattion'] = np.sign(df_dar_merge['correlation']*df_dar_merge['log2FC'])

In [306]:
df_dar_merge['Direction'] = np.where(df_dar_merge['regulattion'] > 0, 'Up', 'Down')

In [322]:
df_ct=pd.crosstab(df_dar_merge['status'], df_dar_merge['Direction'])

In [323]:
df_ct

Direction,Down,Up
status,,
Down,501984,0
Up,0,377254


In [324]:
df_dar_merge.to_csv("/data2st1/junyi/output/atac1112/dar/celltype.L2/MASTNG_dar_corr.csv", index=False)

In [310]:
df_dar_merge

,names,Pr(>Chisq),coef,ci.hi,ci.lo,padjust,celltype.L2,region,ctname,fdr,...,Gene,log2FC,Direction,Neurotransmitter,Gene_name,FDR,gene_coaccess,correlation,gene_back,regulattion
0,chr10:100456562-100457063,4.696654e-03,0.002657,0.014730,-0.009417,1.000000e+00,HPF_CA3_Glut,HPF,HPF_CA3_Glut,8.837957e-03,...,chr10:100456562-100457063,0.002657,Up,Glut,chr10:100456562-100457063,8.837957e-03,Kitl,0.266765,Tmtc3,1.0
1,chr10:100456562-100457063,4.696654e-03,0.002657,0.014730,-0.009417,1.000000e+00,HPF_CA3_Glut,HPF,HPF_CA3_Glut,8.837957e-03,...,chr10:100456562-100457063,0.002657,Up,Glut,chr10:100456562-100457063,8.837957e-03,4930430F08Rik,0.221632,Tmtc3,1.0
2,chr10:100487209-100487710,4.226688e-12,0.080716,0.103431,0.058001,1.371518e-07,HPF_CA3_Glut,HPF,HPF_CA3_Glut,1.878792e-09,...,chr10:100487209-100487710,0.080716,Up,Glut,chr10:100487209-100487710,1.878792e-09,Cep290,0.225227,Tmtc3,1.0
3,chr10:100487209-100487710,4.226688e-12,0.080716,0.103431,0.058001,1.371518e-07,HPF_CA3_Glut,HPF,HPF_CA3_Glut,1.878792e-09,...,chr10:100487209-100487710,0.080716,Up,Glut,chr10:100487209-100487710,1.878792e-09,Cep290,0.225227,Tmtc3,1.0
4,chr10:100487209-100487710,6.084578e-06,0.034846,0.050596,0.019097,5.822942e-02,AMY_Meis1_Abi3bp_Glut,AMY,AMY_Meis1_Abi3bp_Glut,1.621989e-04,...,chr10:100487209-100487710,0.034846,Up,Glut,chr10:100487209-100487710,1.621989e-04,Cep290,0.225227,Tmtc3,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
879233,chrX:73824735-73825236,4.155088e-03,0.013819,0.025836,0.001803,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,1.786323e-02,...,chrX:73824735-73825236,0.013819,Up,Glut,chrX:73824735-73825236,1.786323e-02,Gdi1,0.273216,Pdzd4,1.0
879234,chrX:73824735-73825236,4.155088e-03,0.013819,0.025836,0.001803,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,1.786323e-02,...,chrX:73824735-73825236,0.013819,Up,Glut,chrX:73824735-73825236,1.786323e-02,Idh3g,0.262363,Pdzd4,1.0
879235,chrX:73824735-73825236,4.155088e-03,0.013819,0.025836,0.001803,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,1.786323e-02,...,chrX:73824735-73825236,0.013819,Up,Glut,chrX:73824735-73825236,1.786323e-02,Pnck,0.243705,Pdzd4,1.0
879236,chrX:99505795-99506296,6.136253e-03,0.013835,0.026267,0.001404,1.000000e+00,AMY_Zbtb7c_Vwa5b1_Glut,AMY,AMY_Zbtb7c_Vwa5b1_Glut,2.356687e-02,...,chrX:99505795-99506296,0.013835,Down,Glut,chrX:99505795-99506296,2.356687e-02,Stard8,-0.471481,Pja1,-1.0
